# HaloGTF: velocity-dependent production API

This notebook demonstrates configuration, reusable initial conditions, a short
fluid run, and reading/plotting its outputs. It runs only 100 steps by default;
this is an API demonstration, not a collapse-time or convergence experiment.
Install the package with `pip install -e .` from the repository root first.
See the [README](../README.md) and [complete output reference](../docs/output-reference.md).


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import halogtf as gtf
from halogtf.io.read import load_snapshot_bundle


## Configure before constructing the state

The configuration groups are `init`, `grid`, `scatter`, `num`, `stop`, `io`,
and `cosmo`. The old `sim` and `prec` groups are now `scatter` and `num`.
The halo below matches the truncated-NFW calibration setup: virial mass
1e12 Msun/h, concentration 30, and truncation parameter Zt=0.00807.

`scatter.c` is the coefficient called beta in the calibration discussion.
The adopted `(alph, c)=(0.57, 0.86)` defaults are for Rutherford scattering
and normalized second-order SMFP transport. First-order and Møller are
available but do not yet have a separate calibration from this study.


In [ ]:
work_dir = Path.cwd() / "demo_outputs"
work_dir.mkdir(exist_ok=True)
config = gtf.Config(
    init=("truncated_nfw", {"Mvir": 1e12, "cvir": 30.0, "Zt": 0.00807}),
    scatter={"model": "rutherford", "sigma_m_0": 10.0, "w": 50.0,
             "smfp_order": 2, "alph": 0.57, "c": 0.86},
    num={"eps_du": 1e-4},
    stop={"t_halt": 1000.0, "rho0_halt": 1e5},
    io={"base_dir": str(work_dir), "overwrite": False, "nlog": 100,
        "chatter": False},
)
config


`sigma_m_0` is in cm²/g and `w` is in km/s. Set `w=float("inf")`
for hard-sphere scattering with the same amplitude, or choose
`model="moller"` for that angular prescription. Both have unit transport
factors in the infinite-w limit. The second-order SMFP factor already
includes the 45/44 normalization needed to keep `b=1.3847` in its
hard-sphere convention; do not add a correction to `b` or `sigma_m_0`.
The historical baseline can be selected explicitly with `alph=0.5, c=0.9`.

Model numbers are chosen automatically as the first unused number. Each
fresh configuration therefore creates a new model directory. `overwrite=False`
protects an existing directory if you explicitly choose a model number.


## Cache and reuse the initial condition

Initialization can take a minute or two. The IC file stores the initial
profile for reuse across scattering runs. It is not a restart checkpoint.
Use a different filename if changing halo parameters or grid settings; retain
the generating configuration. Loading this file still performs hydrostatic
relaxation. `State.from_dir()` is not implemented.


In [ ]:
ic_path = work_dir / "ic_truncated_nfw_M1e12_c30_Zt000807_defaultgrid.dat"
if not ic_path.exists():
    gtf.State.make_ic_file(config, str(ic_path))
state = gtf.State.from_config(config, ic_filepath=str(ic_path))
model_dir = Path(state.config.io.base_dir) / state.config.io.model_dir
print(model_dir)


## Inspect the new units

Both reference moments use the same fixed **1D dispersion** `0.33*v_s`.
There is no additional sqrt(2) factor: the moment calculation already
performs the relative-velocity thermal average. With
`F=sqrt(K_L_ref*K_S_ref)`, the effective dimensionless amplitude is
`hat_sigma_eff=hat_sigma_0*F`, and `t_s=t0/K_L_ref`, where
`t0=1/(a*sigma_m_0*rho_s*v_s)` in consistent physical units.
Local moments continue to evolve with the fluid temperature.


In [ ]:
for name in ("v_s", "v_ref_over_vs", "w_char", "k_l_ref", "k_s_ref",
             "sigma_m_0_char_hardsphere", "sigma_m_0_char", "t_s"):
    print(f"{name}: {getattr(state.char, name):.8g}")
# v_s is km/s, t_s is seconds; the other displayed quantities are dimensionless.
state.plot_snapshots(snapshots=0, profiles=["rho", "v2", "kn_cond"],
                     filepath=str(model_dir / "initial.png"), show=True)


## Evolve for a bounded number of steps

Each step solves implicit conduction, then restores hydrostatic equilibrium
at fixed shell masses. `v2` is squared 1D dispersion, pressure is `rho*v2`,
and specific internal energy is `1.5*v2`.

Remove `steps=100` for a full run. Alternatively, `run(stoptime=55.0)`
requests an additional duration of 55 in the new time units. Configured
`stop.t_halt` is an absolute time. Despite its name, `run(rho_c=500.0)`
tests **innermost-cell density**, not the mean core density. The configured
`stop.rho0_halt` uses the same central density, with a numerical `time>50`
guard. The first satisfied stopping condition ends the run. Repeated
`run()` calls on the same state continue the evolution.


In [ ]:
state.run(steps=100)


## Read and compare central and mean core density

`rho0` is the innermost-cell density. The boundary `r_c` is where the density
falls to half `rho0`; `rho_c=3*m_c/r_c**3` is the **mean enclosed density**,
not half the central density. Our arrival-time calibration uses the mean
core density. This short run is not expected to reach that target.

Use header names, not column numbers from another code. `time` is in the
new `t_s` units; `time_Gyr` is physical time. Histories are sampled when the
central density changes by `io.drho_tevol`, and at snapshots/endpoints, so a
short run can have only a few saved rows.


In [ ]:
history = np.atleast_1d(np.genfromtxt(model_dir / "time_evolution.txt", names=True))
fig, ax = plt.subplots()
ax.plot(history["time_Gyr"], history["rho0"], label="Central density")
ax.plot(history["time_Gyr"], history["rho_c"], label="Mean core density")
ax.set(xlabel="Time [Gyr]", ylabel="Density / rho_s", yscale="log")
ax.legend()
fig.tight_layout()
fig.savefig(model_dir / "density_history.png")
plt.show()


In [ ]:
state.plot_time_evolution(quantity="kn_cond_c", grid=True, show=True)
state.plot_snapshots(snapshots=[0, -1], profiles=["rho", "v2", "kn_cond"], show=True)
snapshot = load_snapshot_bundle(str(model_dir))


The plotting functions also accept multiple states/configurations or model
numbers; for model numbers, supply `base_dir`. Select actual snapshot indices
from `snapshot_conversion.txt` rather than assuming that a short run writes
hundreds of snapshots. The read-only `load_snapshot_bundle` loads saved
profiles without resuming the simulation.

## Optional animation

Movie generation requires ffmpeg. It is disabled here so that Run All does
not depend on that external program. `parallel=False` renders serially;
`insets=False` disables the default central-density inset.


In [ ]:
make_animation = False
if make_animation:
    state.make_movie(profiles=["rho", "v2"], insets=False, parallel=False)
